In [ ]:
from pathlib import Path
import os
import shutil
import glob
import time
import sys

# 1) Make sure Python can see your OptimizedDataGenerator4.py
sys.path.insert(0, "/home/youeric/PixelML/TestingData")
from OptimizedDataGenerator4 import OptimizedDataGenerator

# source and destination roots
parquet_dir   = "/local/d1/smartpixML/MuonColliderSim/Simulation_Output/"
tf_parent_dir = "/local/d1/smartpixML/filtering_models"

# shared settings
is_directory_recursive = False
file_type              = "parquet"
data_format            = "3D"
normalization          = 1
file_fraction          = 0.8    # fraction for training split
to_standardize         = False
input_shape            = (20, 13, 21)
transpose              = (0, 2, 3, 1)
time_stamps            = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]   # or whatever timestamps you need
x_feature_description  = "all"
filteringBIB           = True

# compute total & train/val file counts once
pattern     = os.path.join(parquet_dir, f"recon{data_format}bib*.{file_type}")
total_files = len(glob.glob(pattern, recursive=is_directory_recursive))
train_count = round(file_fraction * total_files)
val_count   = total_files - train_count

batch_sizes = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]

for bs in batch_sizes:
    folder_name    = f"Batch{bs}_TimeAll"
    records_dir    = os.path.join(tf_parent_dir, folder_name)
    tf_dir_train   = os.path.join(records_dir, "tfrecords_train")
    tf_dir_val     = os.path.join(records_dir, "tfrecords_validation")

    # clean up or create folders
    if os.path.exists(records_dir):
        shutil.rmtree(records_dir)
    os.makedirs(tf_dir_train, exist_ok=True)
    os.makedirs(tf_dir_val,   exist_ok=True)

    print(f"\n>> Writing TFRecords for batch size {bs} into '{folder_name}'")

    # --- training split ---
    start = time.time()
    train_gen = OptimizedDataGenerator(
        data_directory_path    = parquet_dir,
        is_directory_recursive = is_directory_recursive,
        file_type              = file_type,
        data_format            = data_format,
        batch_size             = bs,
        to_standardize         = to_standardize,
        normalization          = normalization,
        file_count             = train_count,
        input_shape            = input_shape,
        transpose              = transpose,
        time_stamps            = time_stamps,
        tf_records_dir         = tf_dir_train,
        x_feature_description  = x_feature_description,
        filteringBIB           = filteringBIB,
    )
    # iterate to trigger write
    print(f"--- Training TFRecords written in {time.time() - start:.1f}s")

    # --- validation split ---
    start = time.time()
    val_gen = OptimizedDataGenerator(
        data_directory_path    = parquet_dir,
        is_directory_recursive = is_directory_recursive,
        file_type              = file_type,
        data_format            = data_format,
        batch_size             = bs,
        to_standardize         = to_standardize,
        normalization          = normalization,
        file_count             = val_count,
        files_from_end         = True,
        input_shape            = input_shape,
        transpose              = transpose,
        time_stamps            = time_stamps,
        tf_records_dir         = tf_dir_val,
        x_feature_description  = x_feature_description,
        filteringBIB           = filteringBIB,
    )
    print(f"--- Validation TFRecords written in {time.time() - start:.1f}s")



>> Writing TFRecords for batch size 2 into 'Batch2_TimeAll'
['/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib0.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib1.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib10.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib11.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib12.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib13.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib14.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib15.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib16.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib17.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib18.parquet', '/local/d1/smartpixML/MuonColliderSim/Simulation_Output/recon3Dbib19.pa